In [ ]:
%pip install pandas matplotlib seaborn scikit-learn
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 68.7 MB/s eta 0:00:00a 0:00:01
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [13]:
df = pd.read_csv('customer_churn_dataset.csv')
print(df.head())
print(df.dtypes)

   customer_id  tenure  monthly_charges  total_charges        contract  \
0            1      52            54.20        2818.40  Month-to-month   
1            2      15            35.28         529.20  Month-to-month   
2            3      72            78.24        5633.28  Month-to-month   
3            4      61            80.24        4894.64        One year   
4            5      21            39.38         826.98  Month-to-month   

  payment_method internet_service tech_support online_security  support_calls  \
0         Credit              DSL           No             Yes              1   
1          Debit              DSL           No              No              2   
2          Debit              DSL           No              No              0   
3           Cash            Fiber          Yes             Yes              0   
4            UPI            Fiber           No              No              4   

  churn  
0    No  
1    No  
2    No  
3    No  
4   Yes  
customer

In [14]:
X = df.drop(columns=['customer_id', 'churn'])
y = df['churn']

In [15]:
categorical_cols = X.select_dtypes(include='object').columns.tolist()
print("Columns to encode:", categorical_cols)

X = pd.get_dummies(X, columns=categorical_cols)
print(X.head())

Columns to encode: ['contract', 'payment_method', 'internet_service', 'tech_support', 'online_security']
   tenure  monthly_charges  total_charges  support_calls  \
0      52            54.20        2818.40              1   
1      15            35.28         529.20              2   
2      72            78.24        5633.28              0   
3      61            80.24        4894.64              0   
4      21            39.38         826.98              4   

   contract_Month-to-month  contract_One year  contract_Two year  \
0                     True              False              False   
1                     True              False              False   
2                     True              False              False   
3                    False               True              False   
4                     True              False              False   

   payment_method_Cash  payment_method_Credit  payment_method_Debit  \
0                False                   True         

In [16]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

In [17]:
from sklearn.model_selection import train_test_split

# 20% goes to testing, 80% for training
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training rows: {len(X_train)}, Test rows: {len(X_test)}")

Training rows: 16000, Test rows: 4000


In [18]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
svm     = SVC(probability=True, class_weight="balanced")
rf      = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")

log_reg.fit(X_train, y_train)
svm.fit(X_train, y_train)

for n in [50, 100, 200, 500]:
    rf = RandomForestClassifier(n_estimators=n, random_state=42, class_weight='balanced')
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)
    print(f"n_estimators={n} → Accuracy: {accuracy_score(y_test, preds):.2%}")

n_estimators=50 → Accuracy: 84.25%
n_estimators=100 → Accuracy: 84.15%
n_estimators=200 → Accuracy: 84.45%
n_estimators=500 → Accuracy: 84.42%


In [22]:
from sklearn.metrics import accuracy_score, classification_report

models = {
    "Logistic Regression": log_reg,
    "SVM":                 svm,
    "Random Forest":       rf
}

for name, model in models.items():
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    print(f"\n{'='*40}")
    print(f"  {name}")
    print(f"  Accuracy: {acc:.2%}")
    print(classification_report(y_test, preds, target_names=["No Churn", "Churn"]))


  Logistic Regression
  Accuracy: 72.22%
              precision    recall  f1-score   support

    No Churn       0.84      0.72      0.77      2645
       Churn       0.57      0.74      0.64      1355

    accuracy                           0.72      4000
   macro avg       0.71      0.73      0.71      4000
weighted avg       0.75      0.72      0.73      4000


  SVM
  Accuracy: 80.50%
              precision    recall  f1-score   support

    No Churn       0.84      0.87      0.85      2645
       Churn       0.73      0.68      0.70      1355

    accuracy                           0.81      4000
   macro avg       0.78      0.77      0.78      4000
weighted avg       0.80      0.81      0.80      4000


  Random Forest
  Accuracy: 84.42%
              precision    recall  f1-score   support

    No Churn       0.85      0.93      0.89      2645
       Churn       0.83      0.68      0.75      1355

    accuracy                           0.84      4000
   macro avg       0.84 

In [23]:
results = pd.DataFrame({
    'Actual':              le.inverse_transform(y_test),
    'Logistic Regression': le.inverse_transform(log_reg.predict(X_test)),
    'SVM':                 le.inverse_transform(svm.predict(X_test)),
    'Random Forest':       le.inverse_transform(rf.predict(X_test))
})

print(results.head(20))

disagreements = results[
    (results['Logistic Regression'] != results['SVM']) |
    (results['SVM'] != results['Random Forest'])
]

print(f"\nRows where models disagree: {len(disagreements)}")
print(disagreements.head(10))

   Actual Logistic Regression  SVM Random Forest
0     Yes                 Yes  Yes           Yes
1      No                  No   No            No
2      No                  No   No            No
3     Yes                 Yes  Yes           Yes
4     Yes                 Yes   No           Yes
5     Yes                 Yes  Yes           Yes
6     Yes                 Yes  Yes           Yes
7     Yes                 Yes  Yes           Yes
8     Yes                 Yes  Yes           Yes
9     Yes                  No   No            No
10     No                  No   No            No
11    Yes                  No   No            No
12     No                  No   No            No
13     No                  No   No            No
14    Yes                  No   No            No
15    Yes                 Yes  Yes           Yes
16     No                  No   No            No
17     No                  No   No            No
18     No                  No   No            No
19     No           